# RAG Retrieval & Ragas Evaluation Notebook

This notebook demonstrates how to load `tw3k_dataset.jsonl`, run **BM25 Lexical Search**, ingest embeddings into **Qdrant Vector Database** (ONNX), and evaluate retrieval quality using **Ragas** (`context_precision` & `context_recall`).

## Step 1: Import Dependencies & Schema

In [ ]:
import json
from pathlib import Path
from src.schema import DocumentChunk
from src.bm25_retriever import BM25Retriever
from src.qdrant_retriever import QdrantRetriever
from src.evaluation import RagasEvaluator

## Step 2 & 3: Load `tw3k_dataset.jsonl` & Ingest as `DocumentChunk` Objects

In [ ]:
dataset_path = Path("tw3k_dataset.jsonl")
chunks = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        chunk_id = data.get("chunk_id", "")
        content = data.get("text", "")
        metadata = {
            "video_id": data.get("video_id"),
            "video_title": data.get("video_title"),
            "formatted_time": data.get("formatted_time"),
            "timestamp_link": data.get("timestamp_link"),
            "channel": data.get("channel")
        }
        if content:
            chunks.append(DocumentChunk(id=chunk_id, content=content, metadata=metadata))

print(f"Successfully loaded {len(chunks)} DocumentChunk objects from {dataset_path.name}.")

## Step 4: Build BM25 Lexical Index

In [ ]:
print(f"Indexing {len(chunks)} chunks with BM25...")
bm25_retriever = BM25Retriever(chunks)
print("BM25 index successfully built!")

## Step 5: Execute Search Queries & Inspect BM25 Results

In [ ]:
queries = [
    "Overexplained tutorial armies generals units",
    "diplomacy coalition alliance vassal warlord",
    "public order corruption commandery tax revenue"
]

for query in queries:
    print(f"\nBM25 QUERY: '{query}'")
    print("=" * 60)
    results = bm25_retriever.search(query, top_k=3)
    for res in results:
        print(f"Rank {res.rank} | BM25 Score: {res.score:.4f} | ID: {res.chunk.id}")
        print(f"Video: {res.chunk.metadata.get('video_title')} ({res.chunk.metadata.get('formatted_time')})")
        print(f"Text snippet: {res.chunk.content[:150]}...")
        print("-" * 60)

## Step 6: Connect to Qdrant Vector Database (Docker / In-Memory)

In [ ]:
qdrant_retriever = QdrantRetriever(collection_name="tw3k_transcripts_db")
print(f"Connected to Qdrant collection: '{qdrant_retriever.collection_name}'")

## Step 7: Embed & Upsert Chunks into Qdrant Vector DB

In [ ]:
print(f"Embedding and indexing {len(chunks)} chunks into Qdrant...")
qdrant_retriever.index_chunks(chunks, batch_size=64)
print("Qdrant vector indexing complete!")

## Step 8: Execute Vector Search Queries in Qdrant

In [ ]:
vector_queries = [
    "How to manage public order and reduce corruption in commanderies?",
    "Cao Cao proxy war credibility tactics",
    "Military strategy for fighting superior enemy army in mountain passes"
]

for query in vector_queries:
    print(f"\nQDRANT VECTOR QUERY: '{query}'")
    print("=" * 65)
    results = qdrant_retriever.search(query, top_k=3)
    for res in results:
        print(f"Rank {res.rank} | Cosine Similarity Score: {res.score:.4f} | ID: {res.chunk.id}")
        print(f"Video: {res.chunk.metadata.get('video_title')} ({res.chunk.metadata.get('formatted_time')})")
        print(f"Text snippet: {res.chunk.content[:150]}...")
        print("-" * 65)

## Step 9: Format Ragas Evaluation Dataset

In [ ]:
eval_samples = [
    {
        "question": "How to lower corruption and improve public order in commanderies?",
        "ground_truth": "Build a Grand Inspectorate and assign administrators with high Authority stats.",
        "retrieved_results": bm25_retriever.search("How to lower corruption and improve public order in commanderies?", top_k=2)
    },
    {
        "question": "What is Cao Cao's unique diplomatic faction mechanic?",
        "ground_truth": "Cao Cao uses Credibility to incite proxy wars and manipulate diplomatic relations with rival warlords.",
        "retrieved_results": qdrant_retriever.search("What is Cao Cao's unique diplomatic faction mechanic?", top_k=2)
    }
]

evaluator = RagasEvaluator()
ragas_dataset = evaluator.format_dataset(eval_samples)
print(f"Ragas Evaluation Dataset created with {len(ragas_dataset)} query samples.")
print(f"Sample Question: {ragas_dataset[0]['question']}")
print(f"Sample Contexts: {ragas_dataset[0]['contexts']}")

## Step 10: Compute Ragas Retrieval Quality Metrics

In [ ]:
bm25_eval = evaluator.evaluate_retriever(eval_samples, retriever_name="BM25 Lexical")
qdrant_eval = evaluator.evaluate_retriever(eval_samples, retriever_name="Qdrant Vector (ONNX)")

print("=== RAGAS EVALUATION METRICS SUMMARY ===")
print(f"[BM25]   Context Precision: {bm25_eval['scores']['context_precision']:.4f} | Context Recall: {bm25_eval['scores']['context_recall']:.4f}")
print(f"[Qdrant] Context Precision: {qdrant_eval['scores']['context_precision']:.4f} | Context Recall: {qdrant_eval['scores']['context_recall']:.4f}")